# 7.17 為什麼先大量讀文章，再練習當助理？


讀過許多食谱，接到「列三種不含花生的點心」仍可能開始介绍食材。文章的合理下一句和使用者需要的交付物不同。前面已實際接過SFT，現在從材料取得與重用看兩階段的理由。

一般文章如「花生可制作糖果」，原文本身提供下一字答案，大量這種學習叫預訓練；問答則需寫理想三項并核對沒花生，接在底座後的學習叫後訓練，SFT是其中一種。先用假設整理成本看差異：文章每筆1單位、核對問答每筆20單位。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
article_count, checked_chat_count = 1000, 50
article_cost, checked_chat_cost = 1, 20
print("文章整理成本", article_count * article_cost)
print("問答整理成本", checked_chat_count * checked_chat_cost)
print("若1000筆都逐筆核對問答", article_count * checked_chat_cost)

輸入是筆數與假設的每筆成本。三行分別是1000、1000、20000：少量需要仔細核對的問答，已經能花掉與大量文章相同的整理預算。這段沒有訓練模型，也沒計算訓練運算費用；它說明為何資料取得方式會影響配方，不能證明「先預訓練一定比較省」或「50筆問答一定夠」。兩階段的實際成效，仍要用固定預算與留出題檢查。

底座是保存的預訓練模型。圖從文章到同一底座，再分給不同任務继續更新；底座可保存重用，分支仍需各自材料和评測。自動取得下一字不表示文章可靠，後訓練也可教任務推理，不只是礼貌。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-07-17-prepost-routes.svg")))

[InstructGPT引言與3.1節](https://arxiv.org/pdf/2203.02155v1)展示這種目標差別與接續流程。眼前成本是假設，不證明兩階段必然更省；已有小世界成绩也未匹配總預算。

練習只把`checked_chat_cost`改成5，先預測問答整理成本變250、全部逐筆核對變5000，再執行核對。接著寫一句這個算例還缺什麼才能選配方：至少缺實際資料品質、訓練成本與留出表現。這些不是用成本比例就能代填的數字。

<details>
<summary>補充：實作約定與原始紀錄</summary>

本教材的[7.12實測](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)讓這條路變得可操作：同一個小世界文字模型，最後十題從對話微調前1題答對，變成微調後7題答對。但它的文字階段只讀同一批45題的材料，沒有學到廣泛知識；與直接SFT的5／10相比，它還多了250次文字更新。因此這份實驗驗證了接續流程與這次成績，沒有隔離產業採用兩階段的因果理由。[19.4](https://birdhackor.github.io/tiny-perceptron-vlm/19.4.html)會把保存與接續用到整合模型，而不是要求你先相信兩階段必勝。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
